In [1]:
import os
import gc
from pathlib import Path
import pyarrow
import fastparquet
from statsmodels.tsa.seasonal import seasonal_decompose
import missingno as msno
import numpy as np
import pandas as pd
import pyreadstat
import seaborn as sns
import matplotlib.pyplot as plt

RUTA_DATA = Path("Datos/mortalidad_estomago_colombia_2008_2024.parquet")
RUTA_POBLACION = Path("Datos/Proyecciones/PROYECCIONES_UNIFICADAS_EDAD.xlsx")
RUTA_AGUA = Path("Datos/Extra/IRCA_DPTO.csv")
RUTA_TABACO = Path("Datos/Extra/Tabaco.xlsx")

In [3]:
# 2) Cargar todos los datos
df_mortalidad = pd.read_parquet(RUTA_DATA)
df_poblacion = pd.read_excel(RUTA_POBLACION)
df_agua = pd.read_csv(RUTA_AGUA)
df_tabaco = pd.read_excel(RUTA_TABACO)

In [4]:
# vista de todos los datos
display(df_mortalidad.head())
display(df_poblacion.head())
display(df_agua.head())
display(df_tabaco.head())
# información de cada dataframe
print("Información de df_mortalidad:")
print(df_mortalidad.info())
print("\nInformación de df_poblacion:")
print(df_poblacion.info())
print("\nInformación de df_agua:")
print(df_agua.info())
print("\nInformación de df_tabaco:")
print(df_tabaco.info())

,Departamento_Defuncion,Municipio_Defuncion,Area_Defuncion,Anio_Defuncion,Mes_Defuncion,Sexo,Grupo_Edad_Detallado,Estado_Civil,Departamento_Residencia,Municipio_Residencia,...,Accidente_Trabajo_Enf_Prof,Ocupacion_Habitual,Grupo_Etnico,Entidad_Administradora_Salud,Clase_EPS,Profesion_Certificador,Causa_Multiple_CIE10,Causa_666_667,Manera_Muerte_Consolidada,Calidad_Diagnostico
0,68,1,1,2008,2,2,19,2,68,1,...,9,,9,9,SIN INFORMACION,1,R092/C80X/C169,201,0,2
1,68,1,1,2008,2,1,18,1,68,1,...,9,,9,999,,1,C169*E43X,201,0,2
2,68,1,1,2008,2,1,20,2,68,276,...,9,,9,2,SIN INFORMACION,1,I469/K729/C169/C80X*D649,201,0,2
3,68,229,1,2008,2,2,20,3,68,229,...,9,,9,2,SIN INFORMACION,1,E889/D489/C169,201,0,2
4,54,1,1,2008,1,1,22,2,54,1,...,9,,9,2,SIN INFORMACION,1,C169/C80X,201,0,2


,DP,DPNOM,AÑO,Hombres_0,Hombres_1,Hombres_2,Hombres_3,Hombres_4,Hombres_5,Hombres_6,...,Total_94,Total_95,Total_96,Total_97,Total_98,Total_99,Total_100,Total Hombres,Total Mujeres,Total general
0,5,Antioquia,2008,46587,47236,47994,48575,49430,50414,51430,...,1137,914,721,571,458,370,1045,2730505,2931594,5662099
1,5,Antioquia,2009,46052,46607,47289,48058,48671,49523,50503,...,1190,966,778,617,495,402,1122,2764235,2966142,5730377
2,5,Antioquia,2010,45638,46107,46698,47390,48157,48793,49640,...,1256,1017,825,669,536,439,1213,2798757,3001403,5800160
3,5,Antioquia,2011,45327,45709,46217,46814,47502,48263,48920,...,1327,1076,873,714,583,476,1318,2833245,3035946,5869191
4,5,Antioquia,2012,45090,45409,45826,46336,46927,47606,48358,...,1401,1142,930,757,628,518,1435,2866567,3069349,5935916


,DepartamentoCodigo,Departamento,Año,IRCA,Nivel de riesgo,IRCAurbano,Nivel de riesgo urbano,IRCArural,Nivel de riesgo rural
0,5,Antioquia,"2,024",5.2,Bajo riesgo,2.4,Sin riesgo,15.3,Riesgo medio
1,5,Antioquia,"2,007",7.8,Riesgo bajo,7.9,Riesgo bajo,5.5,Riesgo bajo
2,5,Antioquia,"2,008",4.4,Sin riesgo,4.4,Sin riesgo,4.7,Sin riesgo
3,5,Antioquia,"2,009",5.4,Riesgo bajo,5.5,Riesgo bajo,5.0,Sin riesgo
4,5,Antioquia,"2,010",16.5,Riesgo medio,10.4,Riesgo bajo,31.0,Riesgo medio


,Código DANE,Departamento,Prevalencia Vida (%)
0,91,Amazonas,25.7
1,5,Antioquia,31.1
2,81,Arauca,33.6
3,88,Archipiélago de San Andres y Providencia,7.9
4,8,Atlántico,24.5


Información de df_mortalidad:
<class 'pandas.DataFrame'>
RangeIndex: 85043 entries, 0 to 85042
Data columns (total 34 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   Departamento_Defuncion        85043 non-null  category
 1   Municipio_Defuncion           85043 non-null  category
 2   Area_Defuncion                85043 non-null  category
 3   Anio_Defuncion                85043 non-null  int16   
 4   Mes_Defuncion                 85043 non-null  category
 5   Sexo                          85043 non-null  category
 6   Grupo_Edad_Detallado          85043 non-null  category
 7   Estado_Civil                  85043 non-null  category
 8   Departamento_Residencia       85043 non-null  category
 9   Municipio_Residencia          85043 non-null  category
 10  Sitio_Defuncion               85043 non-null  category
 11  Causa_Basica_CIE10            85043 non-null  category
 12  Certificador               